# Laboratorio 7. Spark MLlib

CC3066 Data Science, Universidad del Valle de Guatemala, segundo semestre 2026

Este notebook trabaja con las bases de Personas de la ENEIC del INE. Los cuatro trimestres de 2025 sirven para el desarrollo y el primer trimestre de 2026 queda reservado para la prueba final.

Esta primera entrega cubre la sección de análisis exploratorio y segmentación. El notebook se ejecuta de principio a fin sin depender de variables creadas a mano.

Los resultados describen a los registros analizados. No son estimaciones oficiales de la población de Guatemala.

## 0. Configuración del entorno

Se usa Spark 3.5 en modo local. Los archivos de Excel se leen con pandas y openpyxl, uno a la vez, y se convierten a Parquet. Desde ese punto todo el procesamiento ocurre en Spark. A pandas solo pasan tablas agregadas o muestras de hasta 5,000 registros para graficar.

In [ ]:
import warnings
from functools import reduce
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import load_workbook

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Lab07-ENEIC")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 200)
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Colores de las gráficas
AZUL, NARANJA, AQUA, AMARILLO, MAGENTA = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"
COLORES = [AZUL, NARANJA, AQUA, AMARILLO, MAGENTA]
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titleweight": "bold",
})

SEMILLA = 42
print("Versión de Spark", spark.version)

### Archivos y variables

Los archivos originales deben estar en `data/raw`. Si los nombres descargados son otros, basta con cambiar la lista `ARCHIVOS`. El período se asigna desde el archivo de procedencia y no desde la columna `TRIMESTRE`.

Los diccionarios de códigos siguen el diccionario de datos de la ENEIC. Antes de ejecutar conviene confirmar las etiquetas de `P03A03A` con el diccionario entregado. Un código que no esté en el diccionario queda como DESCONOCIDO.

In [ ]:
DIR_RAW = Path("data/raw")
DIR_INTERIM = Path("data/interim")
DIR_PROCESSED = Path("data/processed")
for d in (DIR_RAW, DIR_INTERIM, DIR_PROCESSED):
    d.mkdir(parents=True, exist_ok=True)

# periodo_archivo, anio_archivo, trimestre_calendario, nombre del archivo, registros esperados según el enunciado
ARCHIVOS = [
    ("2025T1", 2025, 1, "ENEIC_2025_I_Personas.xlsx", 51588),
    ("2025T2", 2025, 2, "ENEIC_2025_II_Personas.xlsx", 51167),
    ("2025T3", 2025, 3, "ENEIC_2025_III_Personas.xlsx", 51583),
    ("2025T4", 2025, 4, "ENEIC_2025_IV_Personas.xlsx", 49338),
    ("2026T1", 2026, 1, "ENEIC_2026_I_Personas.xlsx", 49843),
]

faltan = [a[3] for a in ARCHIVOS if not (DIR_RAW / a[3]).exists()]
if faltan:
    raise FileNotFoundError(f"No se encontraron estos archivos en {DIR_RAW.resolve()} {faltan}")

# Columnas originales que se conservan y su nombre analítico
COLUMNAS = {
    "P05D01": "salario_mensual",
    "P02A03": "edad",
    "P05C07A": "antiguedad_anios",
    "P05C07B": "antiguedad_meses",
    "P05H01A": "horas_semanales",
    "P03A03A": "nivel_educativo",
    "P05C16": "categoria_ocupacional",
    "DOMINIO": "dominio",
    "OCUPADOS": "ocupado",
    "NUM_HOGAR": "NUM_HOGAR",
    "NUM_PERSONA": "NUM_PERSONA",
    "FACTOR": "FACTOR",
    "ANIO": "ANIO",
    "TRIMESTRE": "TRIMESTRE",
}

DICC_NIVEL_EDUCATIVO = {
    0: "Ninguno",
    1: "Preprimaria",
    2: "Primaria",
    3: "Básico",
    4: "Diversificado",
    5: "Superior",
    6: "Postgrado",
}
DICC_CATEGORIA = {
    1: "Empleado de gobierno",
    2: "Empleado de empresa privada",
    3: "Jornalero o peón",
    4: "Servicio doméstico",
}
DICC_DOMINIO = {
    1: "Urbano metropolitano",
    2: "Resto urbano",
    3: "Rural nacional",
}
CATEGORIAS_ASALARIADAS = list(DICC_CATEGORIA)
CLAVE = ["periodo_archivo", "NUM_HOGAR", "NUM_PERSONA"]

## 1. Carga, armonización y calidad de datos

### 1.1 Estructura de columnas de cada archivo

Antes de leer los datos se revisa el encabezado de cada archivo. La tabla muestra cuántas columnas tiene y en qué posición aparece cada variable seleccionada.

In [ ]:
def leer_encabezado(ruta):
    # Lee solo la primera fila para conocer nombres y posiciones de columnas
    if ruta.suffix.lower() == ".csv":
        return [str(c).strip().upper() for c in pd.read_csv(ruta, nrows=0).columns]
    wb = load_workbook(ruta, read_only=True)
    fila = next(wb.worksheets[0].iter_rows(min_row=1, max_row=1, values_only=True))
    wb.close()
    return [str(c).strip().upper() for c in fila if c is not None]


encabezados = {p: leer_encabezado(DIR_RAW / nombre) for p, _, _, nombre, _ in ARCHIVOS}

posiciones = pd.DataFrame(
    {p: {c: (cols.index(c) if c in cols else None) for c in COLUMNAS} for p, cols in encabezados.items()}
)
posiciones.loc["TOTAL DE COLUMNAS"] = [len(cols) for cols in encabezados.values()]
posiciones

In [ ]:
base = encabezados["2025T1"]
for p, cols in encabezados.items():
    extra = [c for c in cols if c not in base]
    ausentes = [c for c in base if c not in cols]
    mismo_orden = cols[: len(base)] == base
    print(f"{p} tiene {len(cols)} columnas, {len(extra)} que no están en 2025T1 y le faltan {len(ausentes)}. "
          f"Mismo orden que 2025T1 {mismo_orden}")
    if extra:
        print("   Ejemplos de columnas nuevas", extra[:10])

El archivo IV de 2025 trae más columnas que los demás y varias de las variables seleccionadas cambian de posición. Por eso la unión se hace con `unionByName`, que empareja por nombre.

### 1.2 Lectura individual y conversión a Parquet

Cada archivo se lee por separado y solo con las 14 columnas requeridas. Todas llegan como texto porque un mismo código puede venir como número o como texto. El DataFrame de Spark se crea con un esquema explícito y se guarda en Parquet junto con las columnas de procedencia.

In [ ]:
ESQUEMA_TEXTO = T.StructType([T.StructField(c, T.StringType(), True) for c in COLUMNAS])


def leer_archivo(ruta):
    seleccion = lambda c: str(c).strip().upper() in COLUMNAS
    if ruta.suffix.lower() == ".csv":
        pdf = pd.read_csv(ruta, usecols=seleccion, dtype=str, encoding_errors="replace")
    else:
        pdf = pd.read_excel(ruta, usecols=seleccion, dtype=str, engine="openpyxl")
    pdf.columns = [str(c).strip().upper() for c in pdf.columns]
    faltantes = set(COLUMNAS) - set(pdf.columns)
    if faltantes:
        raise ValueError(f"{ruta.name} no tiene las columnas {faltantes}")
    pdf = pdf[list(COLUMNAS)].astype(object)
    return pdf.where(pdf.notna(), None)


conteo_original = []
for periodo, anio, trim, nombre, esperados in ARCHIVOS:
    pdf = leer_archivo(DIR_RAW / nombre)
    sdf = (
        spark.createDataFrame(pdf, schema=ESQUEMA_TEXTO)
        .withColumn("archivo_origen", F.lit(nombre))
        .withColumn("periodo_archivo", F.lit(periodo))
        .withColumn("anio_archivo", F.lit(anio).cast("int"))
        .withColumn("trimestre_calendario", F.lit(trim).cast("int"))
    )
    sdf.write.mode("overwrite").parquet(str(DIR_INTERIM / f"crudo_{periodo}.parquet"))
    conteo_original.append({"periodo_archivo": periodo, "archivo_origen": nombre,
                            "registros_leidos": len(pdf), "registros_enunciado": esperados})
    del pdf

pd.DataFrame(conteo_original)

### 1.3 Armonización de tipos y unión

Los valores se limpian de espacios y el texto vacío se trata como ausente. Las variables numéricas pasan a `double`. Los códigos pasan a entero solo si el valor es un número entero. Los identificadores de hogar y persona se guardan como texto normalizado para no perder valores que no sean numéricos.

La columna `TRIMESTRE` se conserva para auditoría. El período de análisis sale del archivo de procedencia.

In [ ]:
def a_numero(c):
    s = F.trim(F.col(c))
    return F.when((s == "") | s.isNull(), F.lit(None)).otherwise(s.cast("double"))


def a_codigo(c):
    x = a_numero(c)
    es_entero = x.isNotNull() & ~F.isnan(x) & (F.abs(x) < 1e15) & (x == F.floor(x))
    return F.when(es_entero, x.cast("long"))


def a_identificador(c):
    s = F.trim(F.col(c))
    return F.when(s != "", F.coalesce(a_codigo(c).cast("string"), s))


def armonizar(df):
    return df.select(
        "archivo_origen", "periodo_archivo", "anio_archivo", "trimestre_calendario",
        a_codigo("ANIO").cast("int").alias("ANIO"),
        a_codigo("TRIMESTRE").cast("int").alias("TRIMESTRE"),
        a_identificador("NUM_HOGAR").alias("NUM_HOGAR"),
        a_identificador("NUM_PERSONA").alias("NUM_PERSONA"),
        a_numero("FACTOR").alias("FACTOR"),
        a_codigo("OCUPADOS").cast("int").alias("ocupado"),
        a_numero("P02A03").alias("edad"),
        a_numero("P05C07A").alias("antiguedad_anios"),
        a_numero("P05C07B").alias("antiguedad_meses"),
        a_numero("P05H01A").alias("horas_semanales"),
        a_codigo("P03A03A").cast("int").alias("nivel_educativo_cod"),
        a_codigo("P05C16").cast("int").alias("categoria_ocupacional_cod"),
        a_codigo("DOMINIO").cast("int").alias("dominio_cod"),
        a_numero("P05D01").alias("salario_mensual"),
    )


crudos = {p: spark.read.parquet(str(DIR_INTERIM / f"crudo_{p}.parquet")) for p, *_ in ARCHIVOS}

crudo_2025 = reduce(lambda a, b: a.unionByName(b), [crudos[p] for p in ["2025T1", "2025T2", "2025T3", "2025T4"]])
crudo_2026 = crudos["2026T1"]

arm_2025 = armonizar(crudo_2025).cache()
arm_2026 = armonizar(crudo_2026).cache()
print(f"Registros 2025 unidos {arm_2025.count():,}")
print(f"Registros 2026 {arm_2026.count():,}")

Esquema y cinco registros de las columnas seleccionadas.

In [ ]:
arm_2025.printSchema()
arm_2025.limit(5).toPandas()

### 1.4 Auditoría de la columna TRIMESTRE

La tabla cruza el valor original de `TRIMESTRE` con el archivo de procedencia. Restar uno a `TRIMESTRE` no sirve porque en el archivo II de 2025 hay registros con valor 2, igual que en el archivo I.

In [ ]:
arm_2025.unionByName(arm_2026).groupBy("periodo_archivo", "anio_archivo", "trimestre_calendario") \
    .pivot("TRIMESTRE").count().orderBy("periodo_archivo").toPandas() \
    .set_index(["periodo_archivo", "anio_archivo", "trimestre_calendario"]).fillna(0).astype(int)

### 1.5 Datos faltantes antes de aplicar filtros

Se cuenta como faltante un valor nulo o un texto vacío en el archivo original. La columna de no convertibles cuenta valores escritos que no pudieron leerse como número o como código entero.

In [ ]:
def tabla_faltantes(crudo, armonizado, etiqueta):
    total = crudo.count()
    nulos = crudo.select([
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c) for c in COLUMNAS
    ]).first().asDict()
    nombres_arm = {
        "P05D01": "salario_mensual", "P02A03": "edad", "P05C07A": "antiguedad_anios",
        "P05C07B": "antiguedad_meses", "P05H01A": "horas_semanales", "P03A03A": "nivel_educativo_cod",
        "P05C16": "categoria_ocupacional_cod", "DOMINIO": "dominio_cod", "OCUPADOS": "ocupado",
        "NUM_HOGAR": "NUM_HOGAR", "NUM_PERSONA": "NUM_PERSONA", "FACTOR": "FACTOR",
        "ANIO": "ANIO", "TRIMESTRE": "TRIMESTRE",
    }
    no_nulos_arm = armonizado.select([F.count(v).alias(k) for k, v in nombres_arm.items()]).first().asDict()
    filas = []
    for c in COLUMNAS:
        filas.append({
            "variable": c,
            "nombre_analitico": COLUMNAS[c],
            f"faltantes_{etiqueta}": nulos[c],
            f"pct_faltantes_{etiqueta}": 100 * nulos[c] / total,
            f"no_convertibles_{etiqueta}": (total - nulos[c]) - no_nulos_arm[c],
        })
    return pd.DataFrame(filas).set_index(["variable", "nombre_analitico"])


faltantes = tabla_faltantes(crudo_2025, arm_2025, "2025").join(tabla_faltantes(crudo_2026, arm_2026, "2026"))
faltantes

Para distinguir un faltante porque la pregunta no aplica de una respuesta no registrada se repite el conteo dentro de la población a la que sí le corresponden las preguntas laborales. Esa población son personas de 15 años o más, ocupadas y asalariadas.

In [ ]:
poblacion_objetivo = arm_2025.filter(
    (F.col("edad") >= 15) & (F.col("ocupado") == 1) & F.col("categoria_ocupacional_cod").isin(CATEGORIAS_ASALARIADAS)
)
n_total, n_obj = arm_2025.count(), poblacion_objetivo.count()
vars_laborales = ["salario_mensual", "antiguedad_anios", "antiguedad_meses", "horas_semanales"]

comparacion = pd.DataFrame({
    "pct_faltante_todos_los_registros": [100 * arm_2025.filter(F.col(v).isNull()).count() / n_total for v in vars_laborales],
    "pct_faltante_asalariados_15_mas": [100 * poblacion_objetivo.filter(F.col(v).isNull()).count() / n_obj for v in vars_laborales],
}, index=vars_laborales)
print(f"Registros 2025 {n_total:,}. Asalariados ocupados de 15 años o más {n_obj:,}")
comparacion

La mayor parte de los faltantes de salario, antigüedad y horas desaparece al restringir la base a asalariados ocupados. Esos faltantes corresponden a personas a las que la pregunta no se les hace, por ejemplo menores, desocupados o trabajadores por cuenta propia. Lo que queda dentro de la población objetivo sí es una respuesta no registrada.

### 1.6 Filtros de población y de calidad

Los filtros se aplican siempre en el mismo orden, tanto en 2025 como en 2026. Cada registro recibe como motivo de exclusión el primer criterio que incumple. Así cada exclusión se cuenta una sola vez y la suma de pasos cuadra con el total.

Las variables categóricas no se filtran. Los códigos ausentes o fuera del diccionario se etiquetan como DESCONOCIDO. El código educativo 0 significa ninguno y se conserva como categoría válida.

In [ ]:
def no_finito(c):
    return F.col(c).isNull() | F.isnan(F.col(c)) | (F.abs(F.col(c)) == float("inf"))


PASOS_FILTRO = [
    ("01 Edad ausente o no finita", no_finito("edad")),
    ("02 Edad menor de 15 años", F.col("edad") < 15),
    ("03 No ocupado", F.coalesce(F.col("ocupado") != 1, F.lit(True))),
    ("04 No asalariado según P05C16", ~F.coalesce(F.col("categoria_ocupacional_cod").isin(CATEGORIAS_ASALARIADAS), F.lit(False))),
    ("05 Salario ausente o no finito", no_finito("salario_mensual")),
    ("06 Salario igual o menor a cero", F.col("salario_mensual") <= 0),
    ("07 Antigüedad ausente o no finita", no_finito("antiguedad_anios") | no_finito("antiguedad_meses")),
    ("08 Meses no enteros o fuera de 0 a 11", (F.col("antiguedad_meses") != F.floor("antiguedad_meses"))
        | (F.col("antiguedad_meses") < 0) | (F.col("antiguedad_meses") > 11)),
    ("09 Antigüedad negativa", F.col("antiguedad") < 0),
    ("10 Antigüedad mayor que la edad", F.col("antiguedad") > F.col("edad")),
    ("11 Horas ausentes o no finitas", no_finito("horas_semanales")),
    ("12 Horas fuera del rango 0 a 168", (F.col("horas_semanales") <= 0) | (F.col("horas_semanales") > 168)),
]
PASO_REPETICION = "13 Repetición exacta de la clave"


def etiquetar(col_cod, dicc):
    mapa = F.create_map(*[x for k, v in dicc.items() for x in (F.lit(k), F.lit(v))])
    return F.coalesce(mapa[F.col(col_cod)], F.lit("DESCONOCIDO"))


COLUMNAS_CONTENIDO = ["ANIO", "TRIMESTRE", "FACTOR", "ocupado", "edad", "antiguedad_anios", "antiguedad_meses",
                      "horas_semanales", "nivel_educativo_cod", "categoria_ocupacional_cod", "dominio_cod",
                      "salario_mensual"]


def marcar_exclusiones(df):
    motivo = F.when(PASOS_FILTRO[0][1], PASOS_FILTRO[0][0])
    for nombre, condicion in PASOS_FILTRO[1:]:
        motivo = motivo.when(condicion, nombre)
    df = (df
          .withColumn("antiguedad", F.col("antiguedad_anios") + F.col("antiguedad_meses") / 12)
          .withColumn("motivo_exclusion", motivo)
          .withColumn("huella", F.sha2(F.to_json(F.struct(*COLUMNAS_CONTENIDO)), 256)))
    # Entre los registros que pasan los filtros, una fila idéntica a otra con la misma clave se marca como repetición
    ventana = Window.partitionBy(*CLAVE, "huella").orderBy("archivo_origen")
    df = df.withColumn("orden_repeticion", F.when(F.col("motivo_exclusion").isNull(), F.row_number().over(ventana)))
    return df.withColumn(
        "motivo_exclusion",
        F.when(F.col("orden_repeticion") > 1, F.lit(PASO_REPETICION)).otherwise(F.col("motivo_exclusion")),
    )


marcado_2025 = marcar_exclusiones(arm_2025).cache()
marcado_2026 = marcar_exclusiones(arm_2026).cache()

In [ ]:
PERIODOS = [a[0] for a in ARCHIVOS]
FINAL = "Registros después de filtros"


def tabla_exclusiones(marcado):
    conteos = (marcado.groupBy("periodo_archivo", F.coalesce("motivo_exclusion", F.lit(FINAL)).alias("paso"))
               .count().toPandas())
    tabla = conteos.pivot(index="paso", columns="periodo_archivo", values="count")
    orden = [p[0] for p in PASOS_FILTRO] + [PASO_REPETICION, FINAL]
    tabla = tabla.reindex(orden).fillna(0).astype(int)
    tabla.loc["00 Registros antes de filtros"] = tabla.sum()
    return tabla.reindex(["00 Registros antes de filtros"] + orden)


exclusiones = pd.concat([tabla_exclusiones(marcado_2025), tabla_exclusiones(marcado_2026)], axis=1)[PERIODOS]
exclusiones["TOTAL_2025"] = exclusiones[["2025T1", "2025T2", "2025T3", "2025T4"]].sum(axis=1)
exclusiones

In [ ]:
restantes = exclusiones[PERIODOS].copy()
inicio = restantes.iloc[0]
cadena = [inicio]
for paso in restantes.index[1:-1]:
    cadena.append(cadena[-1] - restantes.loc[paso])
restantes_por_paso = pd.DataFrame(cadena, index=["Inicio"] + list(restantes.index[1:-1])).astype(int)
print("Registros que quedan después de cada paso")
restantes_por_paso

Antes y después de los filtros por archivo, con el porcentaje de registros retenidos.

In [ ]:
antes_despues = pd.DataFrame({
    "registros_antes": exclusiones.loc["00 Registros antes de filtros", PERIODOS],
    "registros_despues": exclusiones.loc[FINAL, PERIODOS],
})
antes_despues["pct_retenido"] = 100 * antes_despues["registros_despues"] / antes_despues["registros_antes"]
antes_despues

### 1.7 Validación de códigos categóricos

Se comparan los códigos observados en la población filtrada con el diccionario. Un código ausente o no reconocido pasa a DESCONOCIDO.

In [ ]:
def preparar(marcado):
    return (
        marcado.filter(F.col("motivo_exclusion").isNull())
        .withColumn("nivel_educativo", etiquetar("nivel_educativo_cod", DICC_NIVEL_EDUCATIVO))
        .withColumn("categoria_ocupacional", etiquetar("categoria_ocupacional_cod", DICC_CATEGORIA))
        .withColumn("dominio", etiquetar("dominio_cod", DICC_DOMINIO))
        .select(
            "archivo_origen", "periodo_archivo", "anio_archivo", "trimestre_calendario", "ANIO", "TRIMESTRE",
            "NUM_HOGAR", "NUM_PERSONA", "FACTOR", "ocupado",
            "salario_mensual", "edad", "antiguedad_anios", "antiguedad_meses", "antiguedad", "horas_semanales",
            "nivel_educativo_cod", "nivel_educativo", "categoria_ocupacional_cod", "categoria_ocupacional",
            "dominio_cod", "dominio",
        )
    )


prep_2025 = preparar(marcado_2025).cache()
prep_2026 = preparar(marcado_2026).cache()

for cod, etiqueta in [("nivel_educativo_cod", "nivel_educativo"), ("categoria_ocupacional_cod", "categoria_ocupacional"),
                      ("dominio_cod", "dominio")]:
    t = (prep_2025.groupBy(cod, etiqueta).count().withColumnRenamed("count", "n_2025")
         .join(prep_2026.groupBy(cod, etiqueta).count().withColumnRenamed("count", "n_2026"), [cod, etiqueta], "full")
         .orderBy(cod).toPandas().fillna(0))
    display(t)

### 1.8 Unicidad de la clave periodo_archivo, NUM_HOGAR y NUM_PERSONA

La revisión se hace sobre la base completa antes de filtros y sobre la base preparada. Cuando una clave aparece más de una vez se separan dos casos. Una repetición exacta tiene el mismo contenido en todas las columnas seleccionadas. Un registro en conflicto comparte la clave pero difiere en alguna columna.

In [ ]:
def revisar_unicidad(df, nombre):
    df = df.withColumn("huella", F.sha2(F.to_json(F.struct(*COLUMNAS_CONTENIDO)), 256))
    claves_nulas = df.filter(F.col("NUM_HOGAR").isNull() | F.col("NUM_PERSONA").isNull()).count()
    por_clave = df.groupBy(CLAVE).agg(F.count("*").alias("filas"), F.countDistinct("huella").alias("versiones"))
    dup = por_clave.filter(F.col("filas") > 1).cache()
    r = dup.agg(
        F.count("*").alias("claves_repetidas"),
        F.sum(F.when(F.col("versiones") == 1, 1).otherwise(0)).alias("claves_con_repeticion_exacta"),
        F.sum(F.when(F.col("versiones") > 1, 1).otherwise(0)).alias("claves_en_conflicto"),
        F.sum("filas").alias("filas_involucradas"),
    ).first().asDict()
    r = {k: (v or 0) for k, v in r.items()}
    r.update({"base": nombre, "filas": df.count(), "claves_distintas": por_clave.count(), "claves_nulas": claves_nulas})
    return r, dup


resultados_unicidad, duplicados = [], {}
for nombre, df in [("2025 antes de filtros", arm_2025), ("2026 antes de filtros", arm_2026),
                   ("2025 preparada", prep_2025), ("2026 preparada", prep_2026)]:
    r, dup = revisar_unicidad(df, nombre)
    resultados_unicidad.append(r)
    duplicados[nombre] = dup

pd.DataFrame(resultados_unicidad).set_index("base")[
    ["filas", "claves_distintas", "claves_nulas", "claves_repetidas", "claves_con_repeticion_exacta",
     "claves_en_conflicto", "filas_involucradas"]]

In [ ]:
# Ejemplos de claves repetidas en la base 2025 antes de filtros
dup_2025 = duplicados["2025 antes de filtros"]
if dup_2025.count() == 0:
    print("No hay claves repetidas en 2025")
else:
    print("Claves repetidas por archivo")
    display(dup_2025.groupBy("periodo_archivo").agg(F.count("*").alias("claves"), F.sum("filas").alias("filas")).toPandas())
    muestra_claves = dup_2025.orderBy(F.desc("versiones")).limit(5)
    display(arm_2025.join(muestra_claves.select(CLAVE), CLAVE).orderBy(*CLAVE).toPandas())

Las repeticiones exactas no aportan información nueva. En la base preparada se retiene la primera copia de cada una y las demás se registran en el paso 13 de la tabla de exclusiones. Esto se hace con una ventana explícita y no con `dropDuplicates()`, así el número queda documentado. Los registros en conflicto se conservan porque no hay forma de saber cuál versión es la correcta. Si aparecen, se reportan en la tabla anterior para revisarlos con el diccionario.

### 1.9 Guardado en Parquet

El conjunto de 2025 y el de 2026 se guardan por separado.

In [ ]:
RUTA_2025 = DIR_PROCESSED / "eneic_2025_preparado.parquet"
RUTA_2026 = DIR_PROCESSED / "eneic_2026_preparado.parquet"
prep_2025.write.mode("overwrite").parquet(str(RUTA_2025))
prep_2026.write.mode("overwrite").parquet(str(RUTA_2026))

df25 = spark.read.parquet(str(RUTA_2025)).cache()
df26 = spark.read.parquet(str(RUTA_2026)).cache()
print(f"2025 preparado {df25.count():,} registros. 2026 preparado {df26.count():,} registros")

### 1.10 Respuestas

**¿Por qué IV de 2025 no puede apilarse por posición de columnas con los otros archivos?**

El archivo IV de 2025 tiene 302 columnas y los otros 270. Las columnas adicionales desplazan la posición de varias variables, como se ve en la tabla de la sección 1.1. Una unión por posición pondría, por ejemplo, valores de una pregunta distinta dentro de la columna de salario sin generar error. `unionByName` empareja cada columna por su nombre y evita ese problema.

**¿Qué diferencia existe entre un dato ausente porque la pregunta no corresponde y una respuesta no registrada?**

El cuestionario tiene saltos. A una persona desocupada o que trabaja por cuenta propia no se le pregunta el salario de asalariado, y ese vacío es parte del diseño. Una respuesta no registrada ocurre cuando la pregunta sí aplicaba y no se obtuvo el dato, por ejemplo porque la persona no sabía o no quiso responder. La tabla de la sección 1.5 muestra que el faltante de salario baja al pasar de todos los registros a los asalariados. Solo lo que queda en la población objetivo es falta de respuesta, y no se imputa.

**¿Por qué una persona observada en dos períodos no debe eliminarse como duplicado del conjunto longitudinal?**

La ENEIC tiene un diseño con rotación y una misma persona puede ser entrevistada en varios trimestres. Cada entrevista registra su situación en ese período, con salario, horas o antigüedad que pueden cambiar. Son observaciones distintas. Por eso la clave incluye `periodo_archivo`. Eliminarlas quitaría información y cambiaría la composición de los trimestres. Hay que tomarlo en cuenta al interpretar que el número de filas no es el número de personas distintas.

**¿Por qué el número de registros de la base filtrada no representa a todos los trabajadores del país?**

Primero porque es una muestra y cada registro representa a un número distinto de personas según su `FACTOR` de expansión. Para estimar totales o promedios de la población habría que ponderar cada registro por ese factor y usar el diseño muestral para los errores. Segundo porque los filtros dejan solo a asalariados con salario positivo registrado y datos válidos. Quedan fuera los trabajadores por cuenta propia, los no remunerados y quienes no reportaron salario. Tercero porque una persona puede aparecer en varios trimestres. En este laboratorio el análisis es no ponderado y describe los registros analizados.

## 2. Estadística descriptiva y preguntas de exploración

Todas las estadísticas se calculan en Spark sobre la población analítica completa de 2025. Los percentiles se obtienen con la función exacta `percentile` de Spark SQL.

In [ ]:
VARS_NUM = {
    "salario_mensual": "Salario mensual (Q)",
    "edad": "Edad (años)",
    "antiguedad": "Antigüedad (años)",
    "horas_semanales": "Horas habituales por semana",
}

aggs = []
for c in VARS_NUM:
    aggs += [
        F.count(c).alias(f"{c}|n"),
        F.mean(c).alias(f"{c}|media"),
        F.stddev(c).alias(f"{c}|desv_estandar"),
        F.min(c).alias(f"{c}|minimo"),
        F.max(c).alias(f"{c}|maximo"),
        F.expr(f"percentile({c}, array(0.25, 0.5, 0.75, 0.95))").alias(f"{c}|pct"),
        F.skewness(c).alias(f"{c}|asimetria"),
    ]
fila = df25.agg(*aggs).first().asDict()

descriptivas = []
for c, nombre in VARS_NUM.items():
    p25, p50, p75, p95 = fila[f"{c}|pct"]
    descriptivas.append({
        "variable": nombre, "n": fila[f"{c}|n"], "media": fila[f"{c}|media"], "mediana": p50,
        "desv_estandar": fila[f"{c}|desv_estandar"], "minimo": fila[f"{c}|minimo"], "maximo": fila[f"{c}|maximo"],
        "p25": p25, "p75": p75, "p95": p95, "asimetria": fila[f"{c}|asimetria"],
    })
descriptivas = pd.DataFrame(descriptivas).set_index("variable")
descriptivas

### 2.1 Distribución de registros por categoría ocupacional, nivel educativo y dominio

In [ ]:
def conteo_categoria(df, cod, etiqueta):
    t = df.groupBy(cod, etiqueta).count().orderBy(cod).toPandas()
    t["pct"] = 100 * t["count"] / t["count"].sum()
    return t


fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, (cod, etiqueta, titulo) in zip(axes, [
    ("categoria_ocupacional_cod", "categoria_ocupacional", "Categoría ocupacional"),
    ("nivel_educativo_cod", "nivel_educativo", "Nivel educativo"),
    ("dominio_cod", "dominio", "Dominio"),
]):
    t = conteo_categoria(df25, cod, etiqueta)
    ax.barh(t[etiqueta], t["count"], color=AZUL, edgecolor="white", linewidth=2)
    ax.invert_yaxis()
    for y, (n, p) in enumerate(zip(t["count"], t["pct"])):
        ax.text(n, y, f"  {n:,} ({p:.1f}%)", va="center", fontsize=8)
    ax.set_title(titulo)
    ax.set_xlabel("Registros")
    ax.set_xlim(0, t["count"].max() * 1.35)
    ax.grid(axis="y", visible=False)
fig.suptitle("Registros de la población analítica 2025", y=1.02)
plt.tight_layout()
plt.show()

Las barras muestran los registros de la muestra analítica, no el número de trabajadores del país. Los empleados de empresa privada concentran la mayor parte de los registros. Los niveles educativos con más observaciones y el peso de cada dominio se leen en las etiquetas de cada barra. Las categorías con pocos registros darán estimaciones menos estables en las comparaciones por grupo y en los modelos.

### 2.2 Forma de la distribución del salario

El histograma de la izquierda usa escala lineal y muestra el rango hasta el percentil 99 para que la forma sea visible. El de la derecha usa el logaritmo base 10 del salario y cubre todos los registros. El objetivo de los modelos sigue siendo el salario en quetzales.

In [ ]:
p99 = df25.agg(F.expr("percentile(salario_mensual, 0.99)")).first()[0]
media_sal = descriptivas.loc["Salario mensual (Q)", "media"]
mediana_sal = descriptivas.loc["Salario mensual (Q)", "mediana"]

ancho_lin = p99 / 60
hist_lin = (df25.filter(F.col("salario_mensual") <= p99)
            .groupBy(F.floor(F.col("salario_mensual") / ancho_lin).alias("b")).count().orderBy("b").toPandas())
ancho_log = 0.05
hist_log = (df25.groupBy(F.floor(F.log10("salario_mensual") / ancho_log).alias("b")).count().orderBy("b").toPandas())

fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
ax = axes[0]
ax.bar(hist_lin["b"] * ancho_lin, hist_lin["count"], width=ancho_lin, align="edge", color=AZUL, edgecolor="white", linewidth=0.5)
ax.axvline(mediana_sal, color=NARANJA, lw=2, label=f"Mediana Q{mediana_sal:,.0f}")
ax.axvline(media_sal, color="#52514e", lw=2, ls="--", label=f"Media Q{media_sal:,.0f}")
ax.set_title("Salario mensual en escala lineal, hasta el percentil 99")
ax.set_xlabel("Salario mensual (Q)")
ax.set_ylabel("Registros")
ax.legend()

ax = axes[1]
ax.bar(10 ** (hist_log["b"] * ancho_log), hist_log["count"],
       width=10 ** ((hist_log["b"] + 1) * ancho_log) - 10 ** (hist_log["b"] * ancho_log),
       align="edge", color=AZUL, edgecolor="white", linewidth=0.5)
ax.set_xscale("log")
ax.axvline(mediana_sal, color=NARANJA, lw=2, label="Mediana")
ax.axvline(media_sal, color="#52514e", lw=2, ls="--", label="Media")
ax.set_title("Salario mensual en escala logarítmica, todos los registros")
ax.set_xlabel("Salario mensual (Q), eje en escala logarítmica")
ax.set_ylabel("Registros")
ax.legend()
plt.tight_layout()
plt.show()

asim = descriptivas.loc["Salario mensual (Q)", "asimetria"]
print(f"Coeficiente de asimetría del salario {asim:.2f}")
print(f"La media supera a la mediana por Q{media_sal - mediana_sal:,.0f}, "
      f"es decir {100 * (media_sal / mediana_sal - 1):.1f}% por encima")
print(f"Registros por encima del percentil 99 que no aparecen en el panel lineal "
      f"{df25.filter(F.col('salario_mensual') > p99).count():,}")

**¿El salario presenta una distribución simétrica o asimétrica?**

Es asimétrica con cola a la derecha. La mayoría de registros se concentra en salarios bajos y medios y un grupo pequeño llega a valores varias veces más altos. El coeficiente de asimetría impreso arriba es positivo. En escala logarítmica la forma se acerca más a una campana, lo que indica que las diferencias relativas son más estables que las absolutas.

**¿Qué diferencia existe entre su media y su mediana?**

La media queda por encima de la mediana porque los salarios altos la jalan hacia la derecha. La mediana representa mejor el salario de un registro típico. Para los modelos esto importa porque RMSE da más peso a los errores en esos salarios altos, que se conservan en la base según las instrucciones.

### 2.3 Salario mediano por nivel educativo y por categoría ocupacional

Cada caja va del percentil 25 al 75, la línea interna es la mediana y los bigotes llegan a los percentiles 10 y 90. Todos los percentiles se calculan en Spark sobre la base completa.

In [ ]:
def percentiles_por_grupo(df, cod, etiqueta):
    return (df.groupBy(cod, etiqueta)
            .agg(F.count("*").alias("n"), F.mean("salario_mensual").alias("media"),
                 F.expr("percentile(salario_mensual, array(0.10, 0.25, 0.5, 0.75, 0.90))").alias("p"))
            .orderBy(cod).toPandas())


def cajas(ax, t, etiqueta, titulo):
    stats = [{"label": f"{e}\n(n={n:,})", "whislo": p[0], "q1": p[1], "med": p[2], "q3": p[3], "whishi": p[4]}
             for e, n, p in zip(t[etiqueta], t["n"], t["p"])]
    ax.bxp(stats, showfliers=False, vert=False, patch_artist=True,
           boxprops={"facecolor": AZUL, "alpha": 0.35, "edgecolor": AZUL},
           medianprops={"color": NARANJA, "linewidth": 2.5},
           whiskerprops={"color": "#52514e"}, capprops={"color": "#52514e"})
    ax.invert_yaxis()
    ax.set_title(titulo)
    ax.set_xlabel("Salario mensual (Q)")


t_nivel = percentiles_por_grupo(df25, "nivel_educativo_cod", "nivel_educativo")
t_cat = percentiles_por_grupo(df25, "categoria_ocupacional_cod", "categoria_ocupacional")

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
cajas(axes[0], t_nivel, "nivel_educativo", "Salario por nivel educativo")
cajas(axes[1], t_cat, "categoria_ocupacional", "Salario por categoría ocupacional")
plt.tight_layout()
plt.show()

for t, etiqueta in [(t_nivel, "nivel_educativo"), (t_cat, "categoria_ocupacional")]:
    t = t.assign(mediana=t["p"].str[2])[[etiqueta, "n", "media", "mediana"]]
    display(t)

El salario mediano sube con el nivel educativo. Los registros con educación superior o postgrado tienen medianas mayores que los de ninguno o primaria, y también una dispersión mayor. Entre categorías ocupacionales, el empleo de gobierno muestra la mediana más alta y el servicio doméstico y los jornaleros las más bajas. Estas diferencias son asociaciones. No aíslan el efecto de la educación, porque la categoría, la edad y el dominio cambian al mismo tiempo. Las categorías con pocos registros, como DESCONOCIDO o postgrado, deben leerse con cuidado.

### 2.4 Tamaño de la muestra y salario mediano por trimestre

In [ ]:
por_trimestre = (df25.groupBy("periodo_archivo")
                 .agg(F.count("*").alias("n"), F.mean("salario_mensual").alias("media"),
                      F.expr("percentile(salario_mensual, 0.5)").alias("mediana"))
                 .orderBy("periodo_archivo").toPandas())
por_trimestre["variacion_mediana_pct"] = 100 * por_trimestre["mediana"].pct_change()
display(por_trimestre)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
axes[0].bar(por_trimestre["periodo_archivo"], por_trimestre["n"], color=AZUL, edgecolor="white", linewidth=2)
for x, n in enumerate(por_trimestre["n"]):
    axes[0].text(x, n, f"{n:,}", ha="center", va="bottom", fontsize=9)
axes[0].set_title("Registros de la población analítica")
axes[0].set_ylabel("Registros")
axes[0].grid(axis="x", visible=False)

axes[1].plot(por_trimestre["periodo_archivo"], por_trimestre["mediana"], marker="o", color=AZUL, lw=2, ms=8)
for x, m in enumerate(por_trimestre["mediana"]):
    axes[1].text(x, m, f"Q{m:,.0f}\n", ha="center", va="bottom", fontsize=9)
axes[1].set_title("Salario mediano")
axes[1].set_ylabel("Salario mensual (Q)")
axes[1].set_ylim(0, por_trimestre["mediana"].max() * 1.2)
plt.tight_layout()
plt.show()

El número de registros cambia entre trimestres porque cambian el tamaño de cada archivo y la cantidad de asalariados con datos válidos. La tabla muestra la variación de la mediana entre trimestres. Un cambio pequeño en la mediana puede deberse a la rotación de la muestra y no necesariamente a un cambio en los salarios. Como las cifras no están ponderadas, no deben leerse como la evolución del salario en el país.